# 🚀 RAG System Interactive Demo V2

This notebook provides a robust, interactive demonstration of the Multimodal Enterprise RAG System.

**Improvements in V2:**
- ✅ **Real-time Status Polling**: Waits for document processing to complete before searching.
- ✅ **Robust Error Handling**: Better API error reporting.
- ✅ **Verified Endpoints**: Uses currently active API endpoints.

## 📋 Workflow:
1. **System Health Check** - Verify backend is reachable.
2. **Authentication** - Register/Login a fresh demo user.
3. **Document Upload** - Upload a sample text document.
4. **Processing Wait** - Poll the status API until indexing is complete.
5. **Hybrid Search** - Perform a search to verify retrieval.

## 🛠️ Prerequisites:
- Backend API running at `http://localhost:8000`


In [1]:
import requests
import json
import time
import random
import string
from datetime import datetime
import pandas as pd
from IPython.display import display, HTML, JSON

# Configuration
BASE_URL = "http://localhost:8000"
print(f"📍 API URL: {BASE_URL}")

📍 API URL: http://localhost:8000


## 🏥 1. System Health Check

In [2]:
def check_health():
    try:
        # Try public health endpoint first, fall back to standard
        endpoints = ["/api/search/public/health", "/health"]
        
        for endpoint in endpoints:
            try:
                response = requests.get(f"{BASE_URL}{endpoint}", timeout=5)
                if response.status_code == 200:
                    print(f"✅ System is ONLINE (via {endpoint})")
                    print(json.dumps(response.json(), indent=2))
                    return True
            except requests.exceptions.RequestException:
                continue
        
        print("❌ System appears OFFLINE. Please check docker containers.")
        return False
    except Exception as e:
        print(f"❌ Health check failed: {e}")
        return False

is_healthy = check_health()

✅ System is ONLINE (via /health)
{
  "status": "healthy",
  "version": "1.0.0",
  "environment": "development",
  "timestamp": 1766034762.4722478
}


## 🔐 2. Authentication

In [3]:
def get_auth_headers(token=None):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers

session_data = {}

if is_healthy:
    # Generate random credentials
    rand_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
    email = f"demo_{rand_suffix}@example.com"
    password = "DemoPass123!"
    org_name = f"Demo Org {rand_suffix}"
    
    print(f"👤 Registering new user: {email}")
    
    # Register
    reg_payload = {
        "email": email,
        "password": password,
        "first_name": "Demo",
        "last_name": "User",
        "organization_name": org_name
    }
    
    try:
        reg_resp = requests.post(f"{BASE_URL}/api/v1/auth/register", json=reg_payload)
        if reg_resp.status_code in [200, 201]:
            print("✅ Registration successful")
            
            # Login
            login_payload = {"email": email, "password": password}
            login_resp = requests.post(f"{BASE_URL}/api/v1/auth/login", json=login_payload)
            
            if login_resp.status_code == 200:
                token_data = login_resp.json()
                session_data["token"] = token_data["access_token"]
                print("✅ Login successful - Token acquired")
            else:
                print(f"❌ Login failed: {login_resp.text}")
        else:
            print(f"❌ Registration failed: {reg_resp.text}")
            
    except Exception as e:
        print(f"❌ Auth error: {e}")

👤 Registering new user: demo_eao5pl@example.com
✅ Registration successful
✅ Login successful - Token acquired


## 📄 3. Document Upload
We will create a sample document about **Quantum Entanglement** to clear vector search results.

In [4]:
sample_text = """
# Quantum Entanglement and Superposition

## Core Concepts
Quantum entanglement is a physical phenomenon that occurs when a group of particles are generated, interacted, or share spatial proximity in a way such that the quantum state of each particle of the group cannot be described independently of the state of the others, including when the particles are separated by a large distance.

## Superposition
Quantum superposition is a fundamental principle of quantum mechanics. It states that, much like waves in classical physics, any two (or more) quantum states can be added together ("superposed") and the result will be another valid quantum state; and conversely, that every quantum state can be represented as a sum of two or more other distinct states.
"""

filename = "quantum_basics.txt"
with open(filename, "w") as f:
    f.write(sample_text)

if "token" in session_data:
    print("📤 Uploading document...")
    
    with open(filename, "rb") as f:
        files = {"file": (filename, f, "text/plain")}
        data = {
            "title": "Quantum Mechanics Basics",
            "description": "Intro to entanglement and superposition",
            "processing_priority": "normal"
        }
        
        headers = {"Authorization": f"Bearer {session_data['token']}"}
        try:
            # Note: Do not send Content-Type header with files, requests handles it (multipart/form-data)
            resp = requests.post(
                f"{BASE_URL}/api/v1/files/upload",
                files=files,
                data=data,
                headers=headers
            )
            
            if resp.status_code in [200, 201]:
                doc_data = resp.json()
                session_data["document_id"] = doc_data["document_id"]
                print("✅ Upload successful")
                print(f"🆔 Document ID: {session_data['document_id']}")
                print(f"📊 Initial Status: {doc_data.get('processing_status')}")
            else:
                print(f"❌ Upload failed: {resp.text}")
        except Exception as e:
            print(f"❌ Upload error: {e}")

📤 Uploading document...
✅ Upload successful
🆔 Document ID: 7b03afbb-71f8-404f-9872-052935b0e843
📊 Initial Status: queued


## ⏳ 4. Processing Status (Polling)
**CRITICAL STEP**: The system needs time to OCR, chunk, embed, and index the document. We will poll the status API until it is `completed`.

In [5]:
def poll_status(document_id, token, timeout=60):
    start_time = time.time()
    print(f"⏳ Waiting for document {document_id} to process...")
    
    while (time.time() - start_time) < timeout:
        try:
            resp = requests.get(
                f"{BASE_URL}/api/v2/realtime/documents/{document_id}/status",
                headers=get_auth_headers(token)
            )
            
            if resp.status_code == 200:
                status_data = resp.json()
                current_status = status_data.get("processing_status")
                progress = status_data.get("overall_progress", 0)
                
                print(f"   Status: {current_status.upper()} ({progress:.1f}%)", end="\r")
                
                if current_status == "completed":
                    print(f"\n✅ Processing Complete!")
                    return True
                elif current_status == "failed":
                    print(f"\n❌ Processing Failed: {status_data.get('processing_error')}")
                    return False
            
        except Exception as e:
            print(f"\n⚠️ Polling error: {e}")
            
        time.sleep(2)
        
    print("\n❌ Polling timed out.")
    return False

if "document_id" in session_data:
    is_processed = poll_status(session_data["document_id"], session_data["token"])
else:
    print("❌ No document to check.")
    is_processed = False

⏳ Waiting for document 7b03afbb-71f8-404f-9872-052935b0e843 to process...


KeyboardInterrupt: 

## 🔍 5. Hybrid Search
Now that the document is indexed, we can search for content within it.

In [ ]:
if is_processed:
    query = "fundamental principle of quantum mechanics"
    print(f"🔎 Searching for: '{query}'")
    
    search_payload = {
        "query": query,
        "search_type": "hybrid",
        "limit": 5
    }
    
    try:
        search_resp = requests.post(
            f"{BASE_URL}/api/search/hybrid",
            json=search_payload,
            headers=get_auth_headers(session_data["token"])
        )
        
        if search_resp.status_code == 200:
            results = search_resp.json().get("results", [])
            print(f"✅ Found {len(results)} results:\n")
            
            for i, res in enumerate(results):
                print(f"{i+1}. {res.get('title')} (Score: {res.get('score'):.4f})")
                print(f"   Preview: {res.get('content')[:150]}...\n")
                
            # Display as dataframe for nicer view
            if results:
                df = pd.DataFrame(results)
                display(df[["title", "score", "content"]].head())
        else:
            print(f"❌ Search failed: {search_resp.text}")
            
    except Exception as e:
        print(f"❌ Search error: {e}")
else:
    print("⚠️ Skipping search because document processing was not successful.")

## 🏆 Result Summary

In [ ]:
print("🎉 Demo Complete!")
print("If you saw search results, the RAG system is fully functional.")